# Боталова Валерия
### В работе должно быть

- [х] ТЗ по семи вопросам до SQL: решение сформулировано до анализа
- [х] Выбранная ступень глубины (прикидка, разбор или глубокий разбор) и пара предложений, почему: цена ошибки и можно ли откатить без следа
- [х] Раздел про качество данных: что проверили и что нашли
- [х] Минимум 2 информативных графика (распределения, интервалы и пр.)
- [х] Вывод в виде действия, а не наблюдения
- [х] Ограничения: что не проверили и что могло исказить результат

### Базовая тема (одно решение)
> 20 июля на Android кликабельность `top_search` резко выше обычной. Что это: ошибка трекинга, и тогда чиним логирование, или реальный эффект, и тогда решаем, менять ли формат показа на Android?
Соберите ТЗ, проверьте качество данных, покажите воронку/кликабельность с интервалом и распределением, сформулируйте решение.
### Бонус (необязательно)
Связать вывод с динамикой расхода `top_search` / `top_boost` после 15 июля — только если базовая часть уже закрыта.


In [16]:
from pathlib import Path
import importlib.util
import subprocess
import sys
import warnings
from scipy import stats

# Доустанавливаем недостающие пакеты (нужно только при первом запуске)
_required = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
             "seaborn": "seaborn", "plotly": "plotly", "duckdb": "duckdb"}
_missing = [pkg for mod, pkg in _required.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("Устанавливаю пакеты:", ", ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import duckdb

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.titlesize"] = 12

# Ищем папку data/ с CSV: рядом с ноутбуком, выше по дереву или в подпапке
# (например, если архив распакован в /content в Google Colab)
def find_data_dir(marker: str = "incident_events.csv") -> Path:
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / "data" / marker).exists():
            return base / "data"
    for p in here.glob("**/data/" + marker):
        return p.parent
    raise FileNotFoundError(
        f"Не найдена папка data/ с файлом {marker}. "
        "Откройте ноутбук из распакованной папки adhoc_lecture_handout/."
    )

DATA = find_data_dir()
print("Данные:", DATA)

# Регистрируем CSV как таблицы, чтобы обращаться к ним из SQL
con = duckdb.connect()

def register_csv(name: str, filename: str) -> pd.DataFrame:
    path = DATA / filename
    df = pd.read_csv(path)
    # year/month/day должны остаться строками с ведущими нулями ('07'),
    # иначе фильтр вида day BETWEEN '01' AND '28' сравнит строки неправильно
    for col in ("year", "month", "day"):
        if col in df.columns:
            if col == "year":
                df[col] = df[col].astype("Int64").astype(str)
            else:
                df[col] = df[col].astype("Int64").map(lambda x: f"{int(x):02d}" if pd.notna(x) else None)
    con.register(name, df)
    print(f"{name:28s} {len(df):>8,} строк  <- {filename}")
    return df

impressions = register_csv("impressions", "impressions.csv")
clicks = register_csv("clicks", "clicks.csv")
responses = register_csv("responses", "responses.csv")
product_daily = register_csv("product_daily", "product_daily.csv")
employer_spend = register_csv("employer_spend", "employer_spend.csv")
experiment_clients = register_csv("experiment_clients", "experiment_clients.csv")
incident_events = register_csv("incident_events", "incident_events.csv")

def sql(query: str) -> pd.DataFrame:
    # Выполнить SQL-запрос (в стиле Trino) к синтетическим таблицам
    return con.execute(query).df()

print("\nГотово. Для запросов используйте функцию sql('''...''').")


Данные: C:\Users\Lera\Desktop\fall_2026\work\adhoc_hhschool\data
impressions                    80,000 строк  <- impressions.csv
clicks                          3,837 строк  <- clicks.csv
responses                         421 строк  <- responses.csv
product_daily                     252 строк  <- product_daily.csv
employer_spend                  2,400 строк  <- employer_spend.csv
experiment_clients             12,573 строк  <- experiment_clients.csv
incident_events                25,500 строк  <- incident_events.csv

Готово. Для запросов используйте функцию sql('''...''').


# Артефакт: готовые функции для ad-hoc


In [2]:
from typing import Iterable, Optional, Sequence, Tuple


def wilson_ci(successes: float, n: float, z: float = 1.96) -> Tuple[float, float]:
    # Доверительный интервал Уилсона для доли. Устойчивее нормального приближения
    # на малых выборках и у краёв (доля близка к 0 или 1).
    if n <= 0:
        return (np.nan, np.nan)
    p = successes / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return (max(0.0, center - margin), min(1.0, center + margin))


def proportion_table(
    df: pd.DataFrame,
    group_cols: Sequence[str],
    success_col: str,
    n_col: str,
    rate_name: str = "rate",
) -> pd.DataFrame:
    # Считает долю по группам и добавляет доверительный интервал Уилсона.
    g = df.groupby(list(group_cols), as_index=False).agg(
        successes=(success_col, "sum"),
        n=(n_col, "sum"),
    )
    g[rate_name] = g["successes"] / g["n"]
    cis = g.apply(lambda r: wilson_ci(r["successes"], r["n"]), axis=1)
    g["ci_low"] = cis.map(lambda x: x[0])
    g["ci_high"] = cis.map(lambda x: x[1])
    return g


def period_flag(dates: pd.Series, cutoff: str, before="before", after="after") -> pd.Series:
    # Размечает даты на период до и после точки cutoff.
    d = pd.to_datetime(dates)
    return np.where(d < pd.Timestamp(cutoff), before, after)


def summarize_dq(df: pd.DataFrame, key_cols: Optional[Iterable[str]] = None) -> pd.DataFrame:
    # Быстрая сводка по качеству данных: размер, дубли, пропуски по колонкам,
    # при указании ключа считаем также дубли по ключу.
    rows = []
    rows.append({"проверка": "строк всего", "значение": len(df)})
    rows.append({"проверка": "полных дублей строк", "значение": int(df.duplicated().sum())})
    for c in df.columns:
        rows.append({"проверка": f"пропуски: {c}", "значение": int(df[c].isna().sum())})
    if key_cols:
        key_cols = list(key_cols)
        rows.append({
            "проверка": f"дубли по ключу: {'+'.join(key_cols)}",
            "значение": int(df.duplicated(key_cols).sum()),
        })
    return pd.DataFrame(rows)


def one_pager(
    decision: str,
    context: str,
    what_done: str,
    result: str,
    limits: str,
    action: str,
) -> str:
    # Собирает итоговую справку с выводом наверху.
    parts = [
        "=" * 60,
        "ИТОГОВАЯ СПРАВКА",
        "=" * 60,
        "",
        "Решение:",
        decision,
        "",
        "Контекст:",
        context,
        "",
        "Что сделали:",
        what_done,
        "",
        "Результат:",
        result,
        "",
        "Ограничения:",
        limits,
        "",
        "Рекомендованное действие:",
        action,
        "=" * 60,
    ]
    return "\n".join(parts)


print("Функции загружены: wilson_ci, proportion_table, period_flag, summarize_dq, one_pager")


Функции загружены: wilson_ci, proportion_table, period_flag, summarize_dq, one_pager


> 20 июля на Android кликабельность `top_search` резко выше обычной. Что это: ошибка трекинга, и тогда чиним логирование, или реальный эффект, и тогда решаем, менять ли формат показа на Android?

## 1. ТЗ и глубина:
| Поле ТЗ | Значение |
|---|---|
| Цель | Понять причину резкого роста CTR в 'top_search' на Android 20 июля |
| Гипотеза | H1: Рост CTR вызван ошибкой логирования. H2: Рост отражает реальное изменение поведения пользователей |
| Метрика | CTR |
| Период | 1-20 июля, сегмент Android |
| Сегменты | Android |
| Критерий | Проверка корректности логирования, отсутствия дублей, стабильности числа показов и сопоставление динамики CTR между платформами |
| Решение | При ошибки трекинга - исправить логирование. При подтверждении реального эффекта - исследовать причины роста и оценить целесообразность изменения формата показа |

Цена ошибки - средняя
| Ошибка | Последствия |
|---|---|
| Принять ошибку логирования за реальный эффект | Изменить формат показа на основании некорректных данных, ухудшить пользовательский опыт и потерять конверсию |
| Принять реальный эффект за ошибку логирования | Не заметить удачное изменение поведения пользователей и упустить возможность улучшить продукт |

Частичный след после отката ошибки останется. 
Исправление логирования -> уже записанные некорректные события автоматически не исправятся. Может потребоваться пересчёт метрик.
Изменение формата показа -> последствия для пользователей, конверсии и накопленных данных полностью отменить нельзя.

И, таким образом, можно сделать вывод, что для цели необходимо выбрать среднюю глубину погружения в проблему. 

## 2. Проверка качества данных

In [11]:
display(summarize_dq(clicks, key_cols=["click_id"]))
display(summarize_dq(impressions, key_cols=["impression_id"]))

orphan_clicks = sql('''
    SELECT COUNT(c.click_id) AS orphan_clicks_count
    FROM clicks c
    LEFT JOIN impressions i ON c.impression_id = i.impression_id
    WHERE i.impression_id IS NULL
''')
print("Количество кликов без показов:", orphan_clicks.iloc[0, 0])

dup_impressions = impressions.duplicated(subset=['impression_id']).sum()
print("Дубликатов impression_id в показах:", dup_impressions)

dup_clicks = clicks.duplicated(
    subset=["click_id"]
).sum()

print(
    "Дубликатов click_id:",
    dup_clicks
)

multiple_clicks = sql("""
    SELECT
        impression_id,
        COUNT(DISTINCT click_id) AS clicks_count
    FROM clicks
    WHERE impression_id IS NOT NULL
    GROUP BY impression_id
    HAVING COUNT(DISTINCT click_id) > 1
    ORDER BY clicks_count DESC
""")

print("Показов с несколькими кликами:", len(multiple_clicks))

display(multiple_clicks.head(10))

,проверка,значение
0,строк всего,3837
1,полных дублей строк,0
2,пропуски: click_id,0
3,пропуски: impression_id,0
4,пропуски: event_date,0
5,пропуски: platform,0
6,пропуски: segment,0
7,пропуски: product,0
8,пропуски: vacancy_id,0
9,пропуски: employer_id,0


,проверка,значение
0,строк всего,80000
1,полных дублей строк,0
2,пропуски: impression_id,0
3,пропуски: event_date,0
4,пропуски: platform,0
5,пропуски: segment,0
6,пропуски: product,0
7,пропуски: vacancy_id,0
8,пропуски: employer_id,0
9,пропуски: hhuid,0


Количество кликов без показов: 0
Дубликатов impression_id в показах: 0
Дубликатов click_id: 0
Показов с несколькими кликами: 0


,impression_id,clicks_count


По результатам первичной проверки структурных проблем в данных не обнаружено:
- пропуски в проверенных полях отсутствуют;
- дубликаты click_id и impression_id не обнаружены;
- клики без соответствующих показов отсутствуют.Ошибок в логировании данных нет.

Наивной проверки на мой взгляд достаточно, так как я проверила дополнительно возможные статусы, и не нашла иных вариаций, которые могли бы помешать проверке через функцию.


In [ ]:
daily_metrics = sql('''
WITH clicked_impressions AS (
    SELECT DISTINCT`
        impression_id
    FROM clicks
    WHERE impression_id IS NOT NULL
),
daily_metrics AS (
    SELECT
        i.event_date,
        i.platform,
        i.product,

        COUNT(DISTINCT i.impression_id) AS impressions,

        COUNT(DISTINCT CASE
            WHEN c.impression_id IS NOT NULL
            THEN i.impression_id
        END) AS clicked_impressions

    FROM impressions i

    LEFT JOIN clicked_impressions c
        ON i.impression_id = c.impression_id

    WHERE i.event_date >= '2026-07-01'
      AND i.event_date < '2026-07-21'
      AND i.product = 'top_search'

    GROUP BY
        i.event_date,
        i.platform,
        i.product
)

SELECT
    event_date,
    platform,
    impressions,
    clicked_impressions,

    ROUND(
        100.0 * clicked_impressions /
        NULLIF(impressions, 0),
        2
    ) AS ctr

FROM daily_metrics
ORDER BY event_date, platform;
''')
display(daily_metrics.head())

fig = px.line(
    daily_metrics,
    x='event_date',
    y='ctr',
    color='platform',
    markers=True,
    title='Динамика CTR top_search по платформам',
    labels={
        'event_date': 'Дата',
        'ctr': 'CTR, %',
        'platform': 'Платформа'
    }
)

fig.add_shape(
    type='line',
    x0='2026-07-20',
    x1='2026-07-20',
    y0=0,
    y1=1,
    xref='x',
    yref='paper',
    line=dict(
        color='red',
        width=2,
        dash='dash'
    )
)

fig.add_annotation(
    x='2026-07-20',
    y=1,
    xref='x',
    yref='paper',
    text='Аномалия — 20 июля',
    showarrow=False,
    yshift=15,
    font=dict(color='red')
)

fig.update_layout(
    hovermode='x unified',
    xaxis_tickangle=-45,
    template='plotly_white'
)

fig.show()

,event_date,platform,impressions,clicked_impressions,ctr
0,2026-07-01,ANDROID,599,16,2.67
1,2026-07-01,DESKTOP_WEB,1045,74,7.08
2,2026-07-01,IOS,493,17,3.45
3,2026-07-01,MOBILE_WEB,634,30,4.73
4,2026-07-02,ANDROID,623,21,3.37


Изначальная гипотеза предполагала, что проблема специфична для Android, но по графику можно заметить скачок и у Mobile_Web. У остальных платформ данные относительно стабильны. Для дальнейшего анализа буду исследовать только две платформы и только один день - 20 июля

In [5]:
metrics = daily_metrics[
    daily_metrics["platform"].isin(
        ["ANDROID", "MOBILE_WEB"]
    )
].copy()

metrics["period"] = np.where(
    pd.to_datetime(metrics["event_date"])
    < pd.Timestamp("2026-07-20"),
    "До аномалии",
    "20 июля"
)

display(metrics)

ctr_table = proportion_table(
    df=metrics,
    group_cols=["platform", "period"],
    success_col="clicked_impressions",
    n_col="impressions",
    rate_name="ctr"
)

ctr_table["ctr_pct"] = ctr_table["ctr"] * 100
ctr_table["ci_low_pct"] = ctr_table["ci_low"] * 100
ctr_table["ci_high_pct"] = ctr_table["ci_high"] * 100

display(ctr_table)

fig = px.bar(
    ctr_table,
    x="platform",
    y="ctr_pct",
    color="period",
    barmode="group",
    error_y=(
        ctr_table["ci_high_pct"] -
        ctr_table["ctr_pct"]
    ),
    error_y_minus=(
        ctr_table["ctr_pct"] -
        ctr_table["ci_low_pct"]
    ),
    title="CTR до и во время аномалии 20 июля",
    labels={
        "platform": "Платформа",
        "ctr_pct": "CTR, %",
        "period": "Период"
    }
)

fig.update_layout(template="plotly_white")
fig.show()


,event_date,platform,impressions,clicked_impressions,ctr,period
0,2026-07-01,ANDROID,599,16,2.67,До аномалии
3,2026-07-01,MOBILE_WEB,634,30,4.73,До аномалии
4,2026-07-02,ANDROID,623,21,3.37,До аномалии
7,2026-07-02,MOBILE_WEB,608,28,4.61,До аномалии
8,2026-07-03,ANDROID,683,29,4.25,До аномалии
11,2026-07-03,MOBILE_WEB,677,35,5.17,До аномалии
12,2026-07-04,ANDROID,595,22,3.70,До аномалии
15,2026-07-04,MOBILE_WEB,657,34,5.18,До аномалии
16,2026-07-05,ANDROID,666,21,3.15,До аномалии
19,2026-07-05,MOBILE_WEB,659,35,5.31,До аномалии


,platform,period,successes,n,ctr,ci_low,ci_high,ctr_pct,ci_low_pct,ci_high_pct
0,ANDROID,20 июля,36,383,0.093995,0.068668,0.127386,9.399478,6.866769,12.738568
1,ANDROID,До аномалии,352,10813,0.032553,0.029371,0.036068,3.255341,2.937091,3.606793
2,MOBILE_WEB,20 июля,32,406,0.078818,0.056381,0.109150,7.881773,5.638108,10.915019
3,MOBILE_WEB,До аномалии,531,10768,0.049313,0.045381,0.053566,4.931278,4.538142,5.356560


In [17]:
def compare_ctr(clicks_anomaly, n_anomaly, clicks_before, n_before):

    p1 = clicks_anomaly / n_anomaly
    p2 = clicks_before / n_before

    pooled = (clicks_anomaly + clicks_before) / (n_anomaly + n_before)

    se = np.sqrt(
        pooled * (1 - pooled) *
        (1 / n_anomaly + 1 / n_before)
    )

    z_stat = (p1 - p2) / se

    p_value = 2 * stats.norm.sf(abs(z_stat))

    return {
        "z_stat": z_stat,
        "p_value": p_value,
        "significant": p_value < 0.05
    }


android_daily = daily_metrics[
    daily_metrics["platform"] == "ANDROID"
].copy()

android_daily["ctr"] = (
    android_daily["clicked_impressions"] /
    android_daily["impressions"]
)

baseline = android_daily[
    pd.to_datetime(android_daily["event_date"])
    < pd.Timestamp("2026-07-20")
]

anomaly = android_daily[
    pd.to_datetime(android_daily["event_date"])
    == pd.Timestamp("2026-07-20")
]

baseline_median = baseline["ctr"].median()
baseline_q1 = baseline["ctr"].quantile(0.25)
baseline_q3 = baseline["ctr"].quantile(0.75)

anomaly_ctr = anomaly["ctr"].iloc[0]

test_result = compare_ctr(
    int(anomaly["clicked_impressions"].sum()),
    int(anomaly["impressions"].sum()),
    int(baseline["clicked_impressions"].sum()),
    int(baseline["impressions"].sum())
)

print("Медиана CTR до аномалии:", round(baseline_median * 100, 2), "%")
print("Q1:", round(baseline_q1 * 100, 2), "%")
print("Q3:", round(baseline_q3 * 100, 2), "%")
print("CTR 20 июля:", round(anomaly_ctr * 100, 2), "%")

print(
    "CTR 20 июля выше медианы в",
    round(anomaly_ctr / baseline_median, 2),
    "раз"
)

print("\nРезультат статистического теста:")
print("Z-статистика:", round(test_result["z_stat"], 4))
print("P-value:", test_result["p_value"])
print("Статистически значимое отличие:", test_result["significant"])


fig = px.box(
    baseline,
    y="ctr",
    points="all",
    title="Распределение дневного CTR Android до 20 июля",
    labels={"ctr": "CTR"}
)

fig.update_traces(
    jitter=0.3,
    pointpos=0
)

fig.add_hline(
    y=anomaly_ctr,
    line_dash="dash",
    line_color="red",
    line_width=2,
    annotation_text="CTR 20 июля — 9,4%",
    annotation_position="top right"
)

fig.update_layout(
    yaxis_tickformat=".1%",
    yaxis_title="CTR",
    xaxis_title="",
    template="plotly_white",
    showlegend=False,
    height=550
)

fig.show()



Медиана CTR до аномалии: 3.15 %
Q1: 2.78 %
Q3: 3.81 %
CTR 20 июля: 9.4 %
CTR 20 июля выше медианы в 2.98 раз

Результат статистического теста:
Z-статистика: 6.4607
P-value: 1.0425337190585955e-10
Статистически значимое отличие: True


## Ограничения исследования
1. **Отсутствие разбивки по часам:** Анализ проводился по дням. Если аномалия длилась всего несколько часов (например, из-за кратковременного сбоя релиза), агрегация по суткам сгладила пиковые значения.
2. **Когортный фактор:** Не проверялось, был ли приток пользователей с аномальным CTR следствием рекламной кампании / внешнего трафика (например, новой рассылки), а не ошибки интерфейса или трекинга.
3. **Платформенный шум:** Частичный рост на `MOBILE_WEB` требует отдельной проверки (возможно, это косвенный эффект или совпадение, не связанное напрямую с Android-кликом).

In [ ]:

print(one_pager(
    decision=(
        "Не менять формат top_search до выяснения причины аномального "
        "роста CTR на Android 20 июля."
    ),

    context=(
        "CTR Android 20 июля составил 9.40% против 3.26% "
        "в предыдущий период. Рост также наблюдается на Mobile Web: "
        "7.88% против 4.93%. Поэтому аномалия не обязательно "
        "связана исключительно с Android."
    ),

    what_done=(        
        "Проверены пропуски, дубликаты и клики без показов. "
        "Рассчитана доля уникальных показов с кликом. "
        "Сопоставлена динамика CTR по платформам. "
        "Рассчитаны доверительные интервалы и выполнено "
        "статистическое сравнение долей." "Проверены пропуски, дубли и клики без показов. Рассчитана доля уникальных показов с кликом. Сравнена динамика CTR по платформам, рассчитаны доверительные интервалы."
    ),

    result=(
        "Выявлена аномалия, требующая дополнительной диагностики. Статистическое отличие показателя само по себе не определяет его причину."
    ),

    limits=(
        "Не проверены почасовая динамика, версии приложения, источники трафика и независимость наблюдений. Не установлена причинно-следственная связь."
    ),

    action=(
        "1. Передать задачу разработчикам на проверку логирования кликов Android.\n"
        "2. Проверить изменения мобильного интерфейса и источников трафика.\n"
        "3. Отдельно исследовать Mobile Web.\n"
        "4. До выяснения причины не менять формат показа."
    )
))


ИТОГОВАЯ СПРАВКА

Решение:
Обнаружено существенное повышение CTR top_search на Android 20 июля. Причина аномалии пока не установлена.

Контекст:
20 июля зафиксировано отклонение CTR Android от предыдущих дневных значений. Изменение также наблюдается на Mobile Web, поэтому проблема не обязательно специфична для Android.

Что сделали:
Проверены пропуски, дубли и клики без показов. Рассчитана доля уникальных показов с кликом. Сравнена динамика CTR по платформам, рассчитаны доверительные интервалы.

Результат:
Выявлена аномалия, требующая дополнительной диагностики. Статистическое отличие показателя само по себе не определяет его причину.

Ограничения:
Не проверены почасовая динамика, версии приложения, источники трафика и независимость наблюдений. Не установлена причинно-следственная связь.

Рекомендованное действие:
1. Передать задачу разработчикам на проверку логирования кликов Android.
2. Проверить изменения мобильного интерфейса и источников трафика.
3. Отдельно исследовать Mobile Web

In [8]:

bonus_data = sql("""
    SELECT event_date, product,
           SUM(net_spend) AS total_spend,
           SUM(impressions) AS total_impressions,
           SUM(clicks) AS total_clicks
    FROM product_daily
    WHERE event_date >= '2026-07-01'
      AND event_date <= '2026-07-25'
      AND product IN ('top_search', 'top_boost')
    GROUP BY event_date, product
    ORDER BY event_date, product
""")

bonus_data["event_date"] = pd.to_datetime(bonus_data["event_date"])

bonus_data["ctr"] = np.where(
    bonus_data["total_impressions"] > 0,
    bonus_data["total_clicks"] /
    bonus_data["total_impressions"],
    np.nan
)

bonus_data["cpc"] = np.where(
    bonus_data["total_clicks"] > 0,
    bonus_data["total_spend"] /
    bonus_data["total_clicks"],
    np.nan
)

fig = px.line(
    bonus_data,
    x="event_date",
    y="ctr",
    color="product",
    markers=True,
    title="Динамика CTR top_search и top_boost"
)

fig.add_shape(
    type="line",
    x0=pd.Timestamp("2026-07-15"),
    x1=pd.Timestamp("2026-07-15"),
    y0=0,
    y1=1,
    xref="x",
    yref="paper",
    line=dict(color="red", width=2, dash="dash")
)

fig.add_annotation(
    x=pd.Timestamp("2026-07-15"),
    y=1,
    xref="x",
    yref="paper",
    text="Запуск top_boost",
    showarrow=False,
    yshift=15,
    font=dict(color="red")
)

fig.update_layout(
    yaxis_tickformat=".1%",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()


fig1 = px.line(
    bonus_data,
    x="event_date",
    y="total_spend",
    color="product",
    markers=True,
    title="Динамика расходов (net_spend) после 15 июля",
    labels={
        "event_date": "Дата",
        "total_spend": "Суммарный расход",
        "product": "Продукт"
    }
)

fig1.add_shape(
    type="line",
    x0=pd.Timestamp("2026-07-15"),
    x1=pd.Timestamp("2026-07-15"),
    y0=0,
    y1=1,
    xref="x",
    yref="paper",
    line=dict(color="red", width=2, dash="dash")
)

fig1.add_annotation(
    x=pd.Timestamp("2026-07-15"),
    y=1,
    xref="x",
    yref="paper",
    text="Старт top_boost",
    showarrow=False,
    yshift=15,
    font=dict(color="red")
)

fig1.add_shape(
    type="line",
    x0=pd.Timestamp("2026-07-20"),
    x1=pd.Timestamp("2026-07-20"),
    y0=0,
    y1=1,
    xref="x",
    yref="paper",
    line=dict(color="orange", width=2, dash="dot")
)

fig1.add_annotation(
    x=pd.Timestamp("2026-07-20"),
    y=1,
    xref="x",
    yref="paper",
    text="Аномалия CTR",
    showarrow=False,
    yshift=30,
    font=dict(color="orange")
)

fig1.update_layout(
    template="plotly_white",
    hovermode="x unified"
)

fig1.show()


fig2 = px.line(
    bonus_data,
    x="event_date",
    y="total_impressions",
    color="product",
    markers=True,
    title="Динамика показов (impressions)"
)

fig2.update_layout(
    template="plotly_white",
    hovermode="x unified"
)

fig2.show()


In [9]:
print(one_pager(
    decision="Причина изменений CTR после запуска top_boost 15 июля не установлена. Взаимосвязь между запуском продукта и аномалией CTR не доказана.",
    context="После 15 июля наблюдается снижение расходов top_search примерно с 680–720 тыс. до 560–590 тыс., а также сокращение показов. При этом CTR обоих продуктов продолжает колебаться без очевидной устойчивой зависимости друг от друга.",
    what_done="Проведён визуальный анализ динамики CTR, расходов net_spend и показов impressions для top_search и top_boost. Сопоставлены изменения показателей до и после запуска top_boost.",
    result="Расходы и показы top_search после 15 июля снизились. CTR не демонстрирует однозначного устойчивого изменения непосредственно после запуска top_boost. Рост CTR в отдельные дни не позволяет установить причину аномалии.",
    limits="Графики показывают агрегированную дневную динамику. Нет почасовой детализации, подтверждения перераспределения бюджетов и анализа других факторов, способных повлиять на CTR. Визуальное совпадение изменений не доказывает причинность.",
    action="1. Проверить изменения бюджетов, ставок и настроек рекламных кампаний после 15 июля.\n2. Исследовать динамику CTR, расходов и показов с разбивкой по платформам и часам.\n3. Проверить, связано ли снижение показов top_search с изменением рекламного бюджета.\n4. Не считать запуск top_boost установленной причиной аномалии CTR до получения дополнительных доказательств."
))

ИТОГОВАЯ СПРАВКА

Решение:
Причина изменений CTR после запуска top_boost 15 июля не установлена. Взаимосвязь между запуском продукта и аномалией CTR не доказана.

Контекст:
После 15 июля наблюдается снижение расходов top_search примерно с 680–720 тыс. до 560–590 тыс., а также сокращение показов. При этом CTR обоих продуктов продолжает колебаться без очевидной устойчивой зависимости друг от друга.

Что сделали:
Проведён визуальный анализ динамики CTR, расходов net_spend и показов impressions для top_search и top_boost. Сопоставлены изменения показателей до и после запуска top_boost.

Результат:
Расходы и показы top_search после 15 июля снизились. CTR не демонстрирует однозначного устойчивого изменения непосредственно после запуска top_boost. Рост CTR в отдельные дни не позволяет установить причину аномалии.

Ограничения:
Графики показывают агрегированную дневную динамику. Нет почасовой детализации, подтверждения перераспределения бюджетов и анализа других факторов, способных повлиять на